In [1]:
import os
import glob
import pandas as pd
from pathlib import Path

In [2]:
BASE_DIR = "bgsner"  # ajuste se necessário
REPORTS_DIR = os.path.join(BASE_DIR, "reports_out")
Path(REPORTS_DIR).mkdir(parents=True, exist_ok=True)


# util simples para listar subpastas imediatas
def list_subdirs(path):
    return sorted([p for p in os.listdir(path) if (Path(path) / p).is_dir()])

In [3]:
INS_DIR = os.path.join(BASE_DIR, "insights_out")
splits_insights = list_subdirs(INS_DIR)

In [4]:
def load_insights(base_dir):
    out = {}
    for split in list_subdirs(base_dir):
        sdir = os.path.join(base_dir, split)
        try:
            q = pd.read_csv(os.path.join(sdir, "length_quantiles.csv"))
            q.insert(0, "split", split)
        except FileNotFoundError:
            q = None
        try:
            oov = pd.read_csv(os.path.join(sdir, "oov_rates.csv"))
            oov.insert(0, "split", split)
        except FileNotFoundError:
            oov = None
        try:
            rare = pd.read_csv(os.path.join(sdir, "rare_labels.csv"))
            rare.insert(0, "split", split)
        except FileNotFoundError:
            rare = None
        try:
            tok_te = pd.read_csv(os.path.join(sdir, "tokens_test.csv"))
            tok_te.insert(0, "split", split)
        except FileNotFoundError:
            tok_te = None
        try:
            tok_tr = pd.read_csv(os.path.join(sdir, "tokens_train.csv"))
            tok_tr.insert(0, "split", split)
        except FileNotFoundError:
            tok_tr = None

        out[split] = {
            "quantiles": q,
            "oov": oov,
            "rare": rare,
            "tokens_test": tok_te,
            "tokens_train": tok_tr,
        }
    return out

In [5]:
ins = load_insights(INS_DIR)

In [6]:
# Tabelas consolidadas simples:
ins_quantiles = pd.concat(
    [ins[s]["quantiles"] for s in ins if ins[s]["quantiles"] is not None],
    ignore_index=True,
)
ins_oov = pd.concat(
    [ins[s]["oov"] for s in ins if ins[s]["oov"] is not None], ignore_index=True
)
ins_rare = pd.concat(
    [ins[s]["rare"] for s in ins if ins[s]["rare"] is not None], ignore_index=True
)
ins_tok_te = pd.concat(
    [ins[s]["tokens_test"] for s in ins if ins[s]["tokens_test"] is not None],
    ignore_index=True,
)
ins_tok_tr = pd.concat(
    [ins[s]["tokens_train"] for s in ins if ins[s]["tokens_train"] is not None],
    ignore_index=True,
)

# OOV em formato largo por métrica
ins_oov_wide = ins_oov.pivot(
    index="split", columns="metric", values="value"
).reset_index()

In [7]:
display(ins_oov_wide)

metric,split,oov_rate_test_vs_train,oov_rate_val_vs_train
0,adversarial,0.081496,0.047011
1,heur_len,0.048294,0.051862
2,heur_rare,0.083663,0.046339
3,loc,0.078238,0.061573
4,reverse,0.057697,0.056520
5,semantic,0.083067,0.055986
6,standard,0.041752,0.050006


In [8]:
CD_DIR = os.path.join(BASE_DIR, "class_dist_out")


def load_class_dist(base_dir):
    rows_counts, rows_props, rows_long = [], [], []
    label_vecs_all = []
    for split in list_subdirs(base_dir):
        sdir = os.path.join(base_dir, split)

        # long
        if Path(os.path.join(sdir, "class_distribution_long.csv")).exists():
            df_long = pd.read_csv(os.path.join(sdir, "class_distribution_long.csv"))
            df_long.insert(0, "split_name", split)
            rows_long.append(df_long)

        # counts e props
        if Path(os.path.join(sdir, "counts_pivot.csv")).exists():
            cnt = pd.read_csv(os.path.join(sdir, "counts_pivot.csv"))
            cnt.insert(0, "split_name", split)
            rows_counts.append(cnt)

        if Path(os.path.join(sdir, "props_pivot.csv")).exists():
            pr = pd.read_csv(os.path.join(sdir, "props_pivot.csv"))
            pr.insert(0, "split_name", split)
            rows_props.append(pr)

        # label_vecs.csv (linha por part)
        if Path(os.path.join(sdir, "label_vecs.csv")).exists():
            lv = pd.read_csv(os.path.join(sdir, "label_vecs.csv"))
            lv.insert(0, "split_name", split)
            label_vecs_all.append(lv)

    long_df = pd.concat(rows_long, ignore_index=True) if rows_long else pd.DataFrame()
    counts = (
        pd.concat(rows_counts, ignore_index=True) if rows_counts else pd.DataFrame()
    )
    props = pd.concat(rows_props, ignore_index=True) if rows_props else pd.DataFrame()
    lvecs = (
        pd.concat(label_vecs_all, ignore_index=True)
        if label_vecs_all
        else pd.DataFrame()
    )
    return long_df, counts, props, lvecs


cd_long, cd_counts, cd_props, cd_lv = load_class_dist(CD_DIR)

In [9]:
merged_wide = cd_counts.merge(
    cd_props,
    on=["split_name", "label"],
    suffixes=("_count", "_prop"),
)
merged_wide.to_csv(
    os.path.join(REPORTS_DIR, "class_counts_props_merged_wide.csv"), index=False
)

In [10]:
props_long = cd_props.melt(
    id_vars=["split_name", "label"],
    var_name="part",
    value_name="prop",
)
props_long_sorted = props_long.sort_values(
    ["split_name", "part", "prop"], ascending=[True, True, False]
)

# counts longo para anexar contagem ao top-5
counts_long = cd_counts.melt(
    id_vars=["split_name", "label"],
    var_name="part",
    value_name="count",
)

In [11]:
top5 = (
    props_long_sorted.groupby(["split_name", "part"], group_keys=False)
    .head(5)
    .merge(counts_long, on=["split_name", "label", "part"], how="left")
)

In [12]:
print("== Merged (wide) ==")
display(merged_wide.head())

== Merged (wide) ==


,split_name,label,test_count,train_count,val_count,test_prop,train_prop,val_prop
0,adversarial,BIOZONE,0,13,4,0.000000,0.000213,0.000460
1,adversarial,CHRONOSTRAT,116,1390,217,0.006801,0.022808,0.024943
2,adversarial,LEXICON,1503,4459,636,0.088121,0.073166,0.073103
3,adversarial,O,15437,55082,7843,0.905077,0.903813,0.901494
4,heur_len,BIOZONE,6,9,2,0.000351,0.000148,0.000225


In [13]:
print("\n== Props (long, sorted) ==")
display(props_long_sorted.head(12))


== Props (long, sorted) ==


,split_name,label,part,prop
3,adversarial,O,test,0.905077
2,adversarial,LEXICON,test,0.088121
1,adversarial,CHRONOSTRAT,test,0.006801
0,adversarial,BIOZONE,test,0.000000
31,adversarial,O,train,0.903813
30,adversarial,LEXICON,train,0.073166
29,adversarial,CHRONOSTRAT,train,0.022808
28,adversarial,BIOZONE,train,0.000213
59,adversarial,O,val,0.901494
58,adversarial,LEXICON,val,0.073103


In [14]:
print("\n== Top-5 por split e partição ==")
display(top5)


== Top-5 por split e partição ==


,split_name,label,part,prop,count
0,adversarial,O,test,0.905077,15437
1,adversarial,LEXICON,test,0.088121,1503
2,adversarial,CHRONOSTRAT,test,0.006801,116
3,adversarial,BIOZONE,test,0.000000,0
4,adversarial,O,train,0.903813,55082
...,...,...,...,...,...
79,standard,BIOZONE,train,0.000129,9
80,standard,O,val,0.902852,7565
81,standard,LEXICON,val,0.079842,669
82,standard,CHRONOSTRAT,val,0.016589,139


In [15]:
top5['part'].value_counts()

part
test     28
train    28
val      28
Name: count, dtype: int64

In [16]:
parts = ["train", "val", "test"]

for split in sorted(top5["split_name"].unique()):
    print(f"\n=== {split} ===")
    for part in parts:
        sub = (
            top5[(top5["split_name"] == split) & (top5["part"] == part)]
            .sort_values("prop", ascending=False)
            .head(5)
            .loc[:, ["label", "prop", "count"]]
            .reset_index(drop=True)
        )
        print(f"\n[{part}] top-5")
        try:
            display(sub)  # funciona no Jupyter
        except NameError:
            print(sub.to_string(index=False))  # fallback se display não existir


=== adversarial ===

[train] top-5


,label,prop,count
0,O,0.903813,55082
1,LEXICON,0.073166,4459
2,CHRONOSTRAT,0.022808,1390
3,BIOZONE,0.000213,13



[val] top-5


,label,prop,count
0,O,0.901494,7843
1,LEXICON,0.073103,636
2,CHRONOSTRAT,0.024943,217
3,BIOZONE,0.000460,4



[test] top-5


...

,label,prop,count
0,O,0.903141,62818
1,LEXICON,0.076817,5343
2,CHRONOSTRAT,0.019912,1385
3,BIOZONE,0.000129,9



[val] top-5


,label,prop,count
0,O,0.902852,7565
1,LEXICON,0.079842,669
2,CHRONOSTRAT,0.016589,139
3,BIOZONE,0.000716,6



[test] top-5


,label,prop,count
0,O,0.910221,7979
1,LEXICON,0.066849,586
2,CHRONOSTRAT,0.022701,199
3,BIOZONE,0.000228,2


In [17]:
CWI_DIR = os.path.join(BASE_DIR, "cosine_out")

# tenta usar o resumo pronto; se não existir, empilha de cada split
summary_path = os.path.join(CWI_DIR, "cosine_summary_all_splits.csv")
if Path(summary_path).exists():
    cos_within_all = pd.read_csv(summary_path)
else:
    rows = []
    for split in list_subdirs(CWI_DIR):
        f = os.path.join(CWI_DIR, split, "cosine_all.csv")
        if Path(f).exists():
            df = pd.read_csv(f)
            df.insert(0, "split", split)
            rows.append(df)
    cos_within_all = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

# Tabelas simples:
# - matriz (a,b) por espaço+split_set (labels/words e pares)
if not cos_within_all.empty:
    cos_within_pivot = (
        cos_within_all.assign(pair=lambda d: d["a"] + "_" + d["b"])
        .pivot_table(
            index=["split", "space"],
            columns="pair",
            values="cosine_distance",
            aggfunc="first",
        )
        .reset_index()
    )
else:
    cos_within_pivot = pd.DataFrame()

In [18]:
cos_within_all.query("space == 'words'")[['split', 'a', 'b', 'cosine_distance']]

,split,a,b,cosine_distance
3,standard,train,val,0.056486
4,standard,train,test,0.056337
5,standard,val,test,0.089422
9,heur_len,train,val,0.058031
10,heur_len,train,test,0.033687
11,heur_len,val,test,0.070671
15,heur_rare,train,val,0.046917
16,heur_rare,train,test,0.038369
17,heur_rare,val,test,0.069474
21,adversarial,train,val,0.049899


In [19]:
cos_within_all.query("space == 'labels'")[["split", "a", "b", "cosine_distance"]]

,split,a,b,cosine_distance
0,standard,train,val,0.000013
1,standard,train,test,0.000071
2,standard,val,test,0.000133
6,heur_len,train,val,0.000016
7,heur_len,train,test,0.000014
8,heur_len,val,test,0.000058
12,heur_rare,train,val,0.000063
13,heur_rare,train,test,0.000164
14,heur_rare,val,test,0.000024
18,adversarial,train,val,0.000003


In [20]:
display(cos_within_pivot)

pair,split,space,train_test,train_val,val_test
0,adversarial,labels,0.000289,0.000003,0.000332
1,adversarial,words,0.240819,0.049899,0.264808
2,heur_len,labels,0.000014,0.000016,0.000058
3,heur_len,words,0.033687,0.058031,0.070671
4,heur_rare,labels,0.000164,0.000063,0.000024
5,heur_rare,words,0.038369,0.046917,0.069474
6,loc,labels,0.001147,0.000795,0.000033
7,loc,words,0.184438,0.121129,0.157371
8,reverse,labels,0.006573,0.001316,0.002011
9,reverse,words,0.102025,0.073681,0.067715


In [21]:
cos_within_pivot.query("space == 'labels'")[
    ["split", "train_test", "train_val", "val_test"]
]

pair,split,train_test,train_val,val_test
0,adversarial,0.000289,0.000003,0.000332
2,heur_len,0.000014,0.000016,0.000058
4,heur_rare,0.000164,0.000063,0.000024
6,loc,0.001147,0.000795,0.000033
8,reverse,0.006573,0.001316,0.002011
10,semantic,0.000065,0.001105,0.000635
12,standard,0.000071,0.000013,0.000133


In [22]:
cos_within_pivot.query("space == 'words'")[
    ["split", "train_test", "train_val", "val_test"]
]

pair,split,train_test,train_val,val_test
1,adversarial,0.240819,0.049899,0.264808
3,heur_len,0.033687,0.058031,0.070671
5,heur_rare,0.038369,0.046917,0.069474
7,loc,0.184438,0.121129,0.157371
9,reverse,0.102025,0.073681,0.067715
11,semantic,0.196384,0.233354,0.322634
13,standard,0.056337,0.056486,0.089422


In [23]:
CB_DIR = os.path.join(BASE_DIR, "cosine_between_out")


def safe_read_csv(path):
    return pd.read_csv(path) if Path(path).exists() else None


cos_bw_train_words = safe_read_csv(os.path.join(CB_DIR, "cos_train_words.csv"))
cos_bw_test_words = safe_read_csv(os.path.join(CB_DIR, "cos_test_words.csv"))
cos_bw_train_labels = safe_read_csv(os.path.join(CB_DIR, "cos_train_labels.csv"))
cos_bw_test_labels = safe_read_csv(os.path.join(CB_DIR, "cos_test_labels.csv"))

In [24]:
cos_bw_train_words

,Unnamed: 0,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
0,standard,0.000000,0.001724,0.007182,0.018297,0.012923,0.019930,0.016301
1,heur_len,0.001724,0.000000,0.009192,0.019389,0.014785,0.020906,0.017564
2,heur_rare,0.007182,0.009192,0.000000,0.021185,0.013416,0.022378,0.019149
3,adversarial,0.018297,0.019389,0.021185,0.000000,0.023141,0.025322,0.026424
4,loc,0.012923,0.014785,0.013416,0.023141,0.000000,0.022321,0.033300
5,semantic,0.019930,0.020906,0.022378,0.025322,0.022321,0.000000,0.040819
6,reverse,0.016301,0.017564,0.019149,0.026424,0.033300,0.040819,0.000000


In [25]:
cos_bw_test_words

,Unnamed: 0,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
0,standard,0.000000,0.072083,0.116639,0.258897,0.268747,0.229773,0.139607
1,heur_len,0.072083,0.000000,0.084778,0.194716,0.217505,0.164474,0.097095
2,heur_rare,0.116639,0.084778,0.000000,0.228616,0.238067,0.213879,0.110655
3,adversarial,0.258897,0.194716,0.228616,0.000000,0.266903,0.160173,0.186896
4,loc,0.268747,0.217505,0.238067,0.266903,0.000000,0.220759,0.313452
5,semantic,0.229773,0.164474,0.213879,0.160173,0.220759,0.000000,0.219549
6,reverse,0.139607,0.097095,0.110655,0.186896,0.313452,0.219549,0.000000


In [26]:
cos_bw_train_labels

,Unnamed: 0,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
0,standard,0.000000e+00,1.387529e-07,0.000009,0.000013,0.000052,0.000020,0.000690
1,heur_len,1.387529e-07,0.000000e+00,0.000011,0.000013,0.000055,0.000023,0.000676
2,heur_rare,9.271282e-06,1.119950e-05,0.000000,0.000039,0.000019,0.000003,0.000858
3,adversarial,1.336726e-05,1.297969e-05,0.000039,0.000000,0.000114,0.000063,0.000546
4,loc,5.218727e-05,5.544868e-05,0.000019,0.000114,0.000000,0.000007,0.001114
5,semantic,2.044744e-05,2.262865e-05,0.000003,0.000063,0.000007,0.000000,0.000945
6,reverse,6.895659e-04,6.755812e-04,0.000858,0.000546,0.001114,0.000945,0.000000


In [27]:
cos_bw_test_labels

,Unnamed: 0,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
0,standard,0.000000,1.998121e-05,0.000010,0.000433,0.000333,2.477302e-05,0.003982
1,heur_len,0.000020,0.000000e+00,0.000035,0.000276,0.000515,4.159078e-07,0.003456
2,heur_rare,0.000010,3.523221e-05,0.000000,0.000430,0.000306,3.872411e-05,0.004164
3,adversarial,0.000433,2.756981e-04,0.000430,0.000000,0.001458,2.568525e-04,0.002186
4,loc,0.000333,5.147184e-04,0.000306,0.001458,0.000000,5.361576e-04,0.006594
5,semantic,0.000025,4.159078e-07,0.000039,0.000257,0.000536,0.000000e+00,0.003410
6,reverse,0.003982,3.456383e-03,0.004164,0.002186,0.006594,3.410425e-03,0.000000
